In [ ]:
# CARRITO DE COMPRA

In [ ]:
# Celda 1. Base de datos de productos
import pandas as pd

datos = [
    {'codigo': 'A001', 'nombre': 'Arroz 1 kg', 'categoria': 'Abarrotes', 'precio': 1500, 'stock': 20},
    {'codigo': 'A002', 'nombre': 'Fideos 400 g', 'categoria': 'Abarrotes', 'precio': 900, 'stock': 35},
    {'codigo': 'B001', 'nombre': 'Leche entera 1 L', 'categoria': 'Lácteos', 'precio': 1200, 'stock': 18},
    {'codigo': 'B002', 'nombre': 'Yogur natural', 'categoria': 'Lácteos', 'precio': 850, 'stock': 25},
    {'codigo': 'C001', 'nombre': 'Bebida cola 1.5 L', 'categoria': 'Bebidas', 'precio': 1900, 'stock': 12},
    {'codigo': 'C002', 'nombre': 'Agua mineral 1.5 L', 'categoria': 'Bebidas', 'precio': 1000, 'stock': 30},
    {'codigo': 'D001', 'nombre': 'Jabón líquido', 'categoria': 'Limpieza', 'precio': 2500, 'stock': 10},
    {'codigo': 'D002', 'nombre': 'Esponja', 'categoria': 'Limpieza', 'precio': 700, 'stock': 40}
]

inventario = pd.DataFrame(datos)
carrito = [] #inicializado como lista vacía
display(inventario)

,codigo,nombre,categoria,precio,stock
0,A001,Arroz 1 kg,Abarrotes,1500,20
1,A002,Fideos 400 g,Abarrotes,900,35
2,B001,Leche entera 1 L,Lácteos,1200,18
3,B002,Yogur natural,Lácteos,850,25
4,C001,Bebida cola 1.5 L,Bebidas,1900,12
5,C002,Agua mineral 1.5 L,Bebidas,1000,30
6,D001,Jabón líquido,Limpieza,2500,10
7,D002,Esponja,Limpieza,700,40


In [ ]:
import pandas as pd
#funciones basicas del carrito

def buscar_productos (inventario, texto, categoria):
  df_filtrado = inventario.copy()
  if texto != '':
    df_filtrado = df_filtrado[df_filtrado['nombre'].str.lower().str.contains(texto.lower())]

  if categoria != 'todas':
    df_filtrado = df_filtrado[df_filtrado['categoria'] == categoria]

  return df_filtrado

def agregar_al_carrito (inventario, carrito, codigo, cantidad):
  producto_inventario = inventario[inventario['codigo'] == codigo]
  if len(producto_inventario) == 0:
    return 'no se conoce el producto'
  producto_inventario = producto_inventario.iloc[0]

  if cantidad <= 0:
    return 'la cantidad debe ser mayor que 0'

  cantidad_actual = 0
  for producto in carrito:
    if producto['codigo'] == codigo:
      cantidad_actual = producto['cantidad']
      break

  if cantidad_actual + cantidad > producto_inventario['stock']:
    return 'no hay suficiente stock'

  if cantidad_actual > 0:
    for producto in carrito:
      if producto['codigo'] == codigo:
        producto['cantidad'] += cantidad
  else:
    carrito.append({
        'codigo': producto_inventario['codigo'],
        'nombre': producto_inventario['nombre'],
        'precio': producto_inventario['precio'],
        'cantidad': cantidad
    })
  return 'producto agregado al carrito'

def quitar_del_carrito (carrito, codigo):
  for producto in carrito:
    if producto['codigo'] == codigo:
      carrito.remove(producto)
      return 'producto quitado del carrito'
  return 'producto no encontrado en el carrito'

def calcular_total (carrito):
  total = 0
  for producto in carrito:
    total += producto['precio'] * producto['cantidad']
  return total

def mostrar_carrito (carrito):
  if len(carrito) == 0:
    return 'el carrito esta vacio'
  tabla = pd.DataFrame(carrito)
  tabla['subtotal'] = tabla['precio'] * tabla['cantidad']
  tabla = tabla[['nombre', 'precio', 'cantidad', 'subtotal']]
  return tabla.to_html(index=False) + '<h3> total:$' + str(calcular_total(carrito)) + '</h3>'



In [ ]:
#COMPONENTES DE LA INTERFAZ
import ipywidgets as widgets
from IPython.display import display

widget_titulo = widgets.HTML(value = '<h1>Carrito de Compra - Minimarket</h1><p>Selecciona productos y agrégalos al carrito</p>')
widget_busqueda = widgets.Text(description = 'Producto', placeholder = 'Ej. Arroz')
widget_categoria = widgets.Dropdown(description = 'Categoria', options = ['Todas', 'Abarrotes', 'Bebidas', 'Lácteos', 'Limpieza'], value = 'Todas')
widget_cantidad = widgets.BoundedIntText(description = 'Cantidad:', value = 1, min = 1)
widget_codigo_quitar = widgets.Text(description = 'Código:', placeholder = 'Ej. A001')

In [ ]:
import ipywidgets as widgets

# BOTONES
widget_boton_buscar = widgets.Button(description='Buscar', button_style='info')
widget_boton_agregar = widgets.Button(description='Agregar al Carrito', button_style='success')
widget_boton_mostrar = widgets.Button(description='Ver Carrito', button_style='info')
widget_boton_quitar = widgets.Button(description='Quitar del Carrito', button_style='danger')
widget_boton_vaciar = widgets.Button(description='Vaciar Carrito', button_style='warning')
widget_boton_limpiar = widgets.Button(description='Limpiar', button_style='success')

# ÁREAS DE SALIDA
widget_salida = widgets.HTML(value='')
widget_mensaje = widgets.HTML(value='')


In [ ]:
#CELDA 4 CONEXION DE BOTONES CON LAS FUNCIONES

def accion_limpiar (boton):
  widget_busqueda.value = ''
  widget_categoria.value = 'Todas'
  widget_cantidad.value = 1
  widget_codigo_quitar.value = ''
  widget_salida.value = ''
  widget_mensaje.value = ''

def accion_buscar (boton):
  resultados = buscar_productos (inventario, widget_busqueda.value, widget_categoria.value)
  if resultados.empty:
    widget_salida.value = 'No se encontraron productos'
  else:
    widget_salida.value = resultados.to_html(index=False)

def accion_agregar (boton):
  # Buscamos primero el producto basándonos en los criterios de búsqueda actuales
  resultados = buscar_productos (inventario, widget_busqueda.value, widget_categoria.value)

  if resultados.empty:
    widget_mensaje.value = 'Busca un producto válido antes de agregarlo'
    return

  # Tomamos el primer producto de la búsqueda para agregarlo
  codigo = resultados.iloc[0]['codigo']
  mensaje_resultado = agregar_al_carrito (inventario, carrito, codigo, widget_cantidad.value)
  widget_mensaje.value = mensaje_resultado
  widget_salida.value = mostrar_carrito (carrito)

def accion_mostrar (boton):
  widget_salida.value = mostrar_carrito (carrito)

def accion_quitar (boton):
  widget_mensaje.value = quitar_del_carrito (carrito, widget_codigo_quitar.value)
  widget_salida.value = mostrar_carrito (carrito)

def accion_vaciar (boton):
  carrito.clear()
  widget_mensaje.value = 'Carrito vaciado'
  widget_salida.value = mostrar_carrito (carrito)

widget_boton_buscar.on_click(accion_buscar)
widget_boton_agregar.on_click(accion_agregar)
widget_boton_mostrar.on_click(accion_mostrar)
widget_boton_quitar.on_click(accion_quitar)
widget_boton_vaciar.on_click(accion_vaciar)
widget_boton_limpiar.on_click(accion_limpiar)

In [ ]:
# Organizar y mostrar la interfaz de usuario simplificada y con colores sutiles
from IPython.display import display
import ipywidgets as widgets

# Estilos sutiles y pasteles
estilo_tarjeta_izq = widgets.Layout(
    border='1px solid #e2e8f0',
    padding='18px',
    margin='10px',
    border_radius='12px',
    background_color='#f8fafc',  # Gris azulado tenue
    width='45%'
)

estilo_tarjeta_der = widgets.Layout(
    border='1px solid #e2e8f0',
    padding='18px',
    margin='10px',
    border_radius='12px',
    background_color='#fcfbf7',  # Crema suave
    width='45%'
)

estilo_pantalla = widgets.Layout(
    border='1px solid #cbd5e1',
    padding='18px',
    margin='10px',
    border_radius='12px',
    background_color='#ffffff',
    width='94%'
)

# Configuración estética de botones
widget_boton_buscar.button_style = ''
widget_boton_agregar.button_style = 'info'
widget_boton_limpiar.button_style = ''
widget_boton_mostrar.button_style = 'primary'  # Botón principal destacado
widget_boton_pago.button_style = 'success'    # Botón de éxito sutil

# Dimensionamiento
widget_boton_buscar.layout.width = '30%'
widget_boton_agregar.layout.width = '45%'
widget_boton_limpiar.layout.width = '20%'
widget_boton_mostrar.layout.width = '100%'
widget_boton_pago.layout.width = '100%'
widget_boton_pago.layout.margin = '10px 0px 0px 0px'

# Sección 1: Panel Izquierdo (Buscar y Agregar)
cont_busqueda = widgets.VBox([
    widgets.HTML('<h3 style="color: #475569; font-family: sans-serif; margin-top: 0;">🔍 Buscar Productos</h3>'),
    widget_busqueda,
    widget_categoria,
    widget_cantidad,
    widgets.HTML('<div style="margin-top: 10px;"></div>'),
    widgets.HBox([widget_boton_buscar, widget_boton_agregar, widget_boton_limpiar], layout=widgets.Layout(justify_content='space-between'))
], layout=estilo_tarjeta_izq)

# Sección 2: Panel Derecho (Mi Carrito Simplificado - Solo Ver y Finalizar Compra)
cont_carrito = widgets.VBox([
    widgets.HTML('<h3 style="color: #475569; font-family: sans-serif; margin-top: 0;">🛒 Mi Carrito</h3>'),
    widgets.HTML('<p style="color: #64748b; font-family: sans-serif; font-size: 0.9em;">Visualiza tus artículos actuales o finaliza la compra para aplicar tu descuento.</p>'),
    widget_boton_mostrar,
    widget_boton_pago
], layout=estilo_tarjeta_der)

# Distribución horizontal
interfaz_controles = widgets.HBox([cont_busqueda, cont_carrito], layout=widgets.Layout(justify_content='space-around'))

# Detalle de salida
pantalla_salida = widgets.VBox([
    widgets.HTML('<h4 style="color: #64748b; font-family: sans-serif; margin-top: 0;">📄 Detalle de Operación / Boleta</h4>'),
    widget_mensaje,
    widgets.HTML('<hr style="border: 0; border-top: 1px solid #f1f5f9;" />'),
    widget_salida
], layout=estilo_pantalla)

# Renderizar la interfaz
display(widget_titulo)
display(widgets.VBox([interfaz_controles, pantalla_salida]))

HTML(value='<h1>Carrito de Compra - Minimarket</h1><p>Selecciona productos y agrégalos al carrito</p>')

In [ ]:
# Nueva funcionalidad: Aplicar descuento y Confirmar Pago (Actualizando el Inventario)

widget_boton_pago = widgets.Button(description='Finalizar Compra', button_style='primary')

def accion_pagar(boton):
    global inventario, carrito
    if not carrito:
        widget_mensaje.value = '<p style="color:red;">El carrito está vacío. No hay nada que pagar.</p>'
        return

    # Aplicar un 10% de descuento de forma ilustrativa
    total_original = calcular_total(carrito)
    descuento = total_original * 0.10
    total_final = total_original - descuento

    # Descontar del inventario real
    for item in carrito:
        codigo = item['codigo']
        cantidad = item['cantidad']
        inventario.loc[inventario['codigo'] == codigo, 'stock'] -= cantidad

    # Crear mensaje de éxito
    resumen = f"""
    <h4>🎉 ¡Compra realizada con éxito! 🎉</h4>
    <p>Total original: ${total_original:,.0f}</p>
    <p>Descuento aplicado (10%): -${descuento:,.0f}</p>
    <p><strong>Total pagado: ${total_final:,.0f}</strong></p>
    <p><em>El inventario ha sido actualizado en tiempo real.</em></p>
    """
    widget_mensaje.value = resumen
    carrito.clear() # Vaciar carrito tras la compra
    widget_salida.value = 'El carrito está vacío.'

widget_boton_pago.on_click(accion_pagar)

# Agregamos visualmente el nuevo botón debajo de los anteriores en la interfaz
cont_carrito.children = list(cont_carrito.children) + [widget_boton_pago]
print("¡Botón 'Finalizar Compra' con 10% de descuento agregado con éxito!")

¡Botón 'Finalizar Compra' con 10% de descuento agregado con éxito!
